In [ ]:
import shutil
import sys
from collections import defaultdict

import polars as pl
from Bio import SeqIO

sys.path.insert(0, "..")
from project import get_all, log

In [ ]:
ALL = get_all()  # or get_all(["plants_plt"])
# ALL = {"plants_plt": {"solanaceae": get_info("plants_plt", "solanaceae")}}

for g, subs in ALL.items():
    print(f"{g}: {list(subs)}")

## Gene-pair tables

In [ ]:
for g, subs in ALL.items():
    for subg, info in subs.items():
        p, cfg = info["paths"], info["config"]
        if not p["atlas"].exists():
            log(f"{g}/{subg}: no {p['atlas'].name}, run 01 first; skipping")
            continue

        n_genomes_total = pl.read_csv(p["tsv"], separator="\t").height
        atlas = pl.read_csv(p["atlas"], separator="\t")

        gene_pairs = (
            atlas.group_by("gene_pair")
            .agg(
                pl.col("AN").n_unique().alias("n_genomes"),
                pl.len().alias("igr_count"),
                pl.col("Length").mean().alias("mean_len"),
                pl.col("Length").median().alias("median_len"),
                pl.col("Length").std().alias("std_len"),
                pl.col("Length").quantile(0.25).alias("q1_len"),
                pl.col("Length").quantile(0.5).alias("q2_len"),
                pl.col("Length").quantile(0.75).alias("q3_len"),
                pl.col("Length").quantile(0.99).alias("q99_len"),
                pl.col("Length").min().alias("min_len"),
                pl.col("Length").max().alias("max_len"),
            )
            .with_columns((pl.col("n_genomes") / n_genomes_total).alias("genome_frac"))
            .select(
                "gene_pair",
                "n_genomes",
                "genome_frac",
                pl.exclude("gene_pair", "n_genomes", "genome_frac"),
            )
            .sort("n_genomes", "igr_count", descending=True)
            .with_columns(pl.col(pl.Float32, pl.Float64).round(2))
        )

        # gene pairs that also occur with the opposite polarity
        flipped = (
            gene_pairs.with_columns(
                pl.col("gene_pair")
                .str.replace(r"_[+-]{2}$", "")
                .alias("gene_pair_cleaned")
            )
            .filter(pl.len().over("gene_pair_cleaned") > 1)
            .sort("gene_pair")
        )

        min_genomes = max(cfg["min_genomes"], cfg["min_genome_frac"] * n_genomes_total)
        filtered = gene_pairs.filter(
            pl.col("n_genomes") >= min_genomes,
            pl.col("mean_len") > cfg["min_mean_len"],
        ).sort("std_len")

        p["gene_pairs_dir"].mkdir(parents=True, exist_ok=True)
        gene_pairs.write_csv(p["gene_pairs"], separator="\t")
        flipped.write_csv(p["gene_pairs_flipped"], separator="\t")
        filtered.write_csv(p["gene_pairs_filtered"], separator="\t")
        log(
            f"{g}/{subg}: {gene_pairs.height} gene pairs | {filtered.height} kept"
            f" (n_genomes >= {min_genomes:g} of {n_genomes_total}, mean_len > {cfg['min_mean_len']})"
            f" | {flipped.height} in flipped"
        )

## One IGR FASTA per kept gene pair

In [ ]:
for g, subs in ALL.items():
    for subg, info in subs.items():
        p = info["paths"]
        if not p["gene_pairs_filtered"].exists():
            continue

        atlas = pl.read_csv(p["atlas"], separator="\t")
        kept = pl.read_csv(p["gene_pairs_filtered"], separator="\t")[
            "gene_pair"
        ].to_list()

        # AN -> {IGR ID: gene_pair}, only for the kept gene pairs
        wanted = atlas.filter(pl.col("gene_pair").is_in(kept)).select(
            "AN", pl.col("ID").cast(pl.String), "gene_pair"
        )
        id_to_gp_by_an: dict[str, dict[str, str]] = {}
        for an, igr_id, gp in wanted.iter_rows():
            id_to_gp_by_an.setdefault(an, {})[igr_id] = gp

        # parse each genome's IGR FASTA once, dispatching records to their gene pair
        records_by_gp = defaultdict(list)
        missing = []
        for an, id_to_gp in id_to_gp_by_an.items():
            fasta = p["igr_by_genome"] / f"{an}_intergenic.fasta"
            if not fasta.exists():
                missing.append(an)
                continue
            for rec in SeqIO.parse(fasta, "fasta"):
                gp = id_to_gp.get(rec.id)
                if gp is not None:
                    records_by_gp[gp].append(rec)

        # rebuilt from scratch, so gene pairs that no longer pass the filter don't linger
        out_dir = p["igr_by_pair"]
        if out_dir.exists():
            shutil.rmtree(out_dir)
        out_dir.mkdir(parents=True)
        for gp, records in records_by_gp.items():
            SeqIO.write(records, out_dir / f"gp_{gp}.fasta", "fasta")

        n_found = sum(len(r) for r in records_by_gp.values())
        log(
            f"{g}/{subg}: {len(records_by_gp)}/{len(kept)} FASTAs written | {n_found}/{wanted.height} IGRs found"
            + (
                f" | no IGR FASTA for {len(missing)} ANs: {missing[:5]}"
                if missing
                else ""
            )
        )